# Tema 9 - Laboratorio 8: paquete propio `soporte_ti`

Ejercicio integrado con paquete propio.

## 1. Objetivo

Usar un paquete propio llamado `soporte_ti`, compuesto por varios módulos internos. En este laboratorio todos los bloques ejecutables muestran el código completo dentro de heredocs.

## 2. Revisar y probar la lógica de `soporte_ti.red`

La celda contiene el código esencial de `soporte_ti/red.py` y una pequeña prueba local. En el proyecto real, este código está en `src/soporte_ti/red.py`.

In [ ]:
%%bash
cd ../src
python - <<'PY'
"""
Módulo red del paquete soporte_ti.

Contiene funciones de normalización y validación de servicios de red.
"""


def normalizar_servicio(nombre: str) -> str:
    """Normaliza nombres de servicio para comparación y reporte."""
    return nombre.strip().lower()


def validar_puerto(puerto: int) -> bool:
    """Devuelve True si el puerto está dentro del rango TCP/UDP válido."""
    return 1 <= puerto <= 65535


def validar_ip_simple(ip: str) -> bool:
    """
    Valida una dirección IPv4 de forma básica.

    Reglas:
        - Debe tener cuatro partes separadas por puntos.
        - Cada parte debe ser numérica.
        - Cada número debe estar entre 0 y 255.
    """
    partes = ip.split(".")

    if len(partes) != 4:
        return False

    for parte in partes:
        if not parte.isdigit():
            return False

        numero = int(parte)
        if numero < 0 or numero > 255:
            return False

    return True


def comprobar_servicio(servicio: dict) -> dict:
    """Normaliza y valida un registro de servicio."""
    nombre = normalizar_servicio(servicio["nombre"])
    ip = servicio["ip"]
    puerto = servicio["puerto"]

    return {
        "nombre": nombre,
        "ip": ip,
        "puerto": puerto,
        "ip_valida": validar_ip_simple(ip),
        "puerto_valido": validar_puerto(puerto),
    }


def comprobar(servicios: list[dict]) -> list[dict]:
    """Comprueba una lista de servicios y devuelve resultados normalizados."""
    resultados = []

    for servicio in servicios:
        resultado = comprobar_servicio(servicio)
        resultados.append(resultado)

    return resultados


servicios = [
    {"nombre": " SSH ", "ip": "10.0.0.10", "puerto": 22},
    {"nombre": "api", "ip": "ip_invalida", "puerto": 8080},
    {"nombre": "backup", "ip": "10.0.0.40", "puerto": 70000},
]

for resultado in comprobar(servicios):
    print(resultado)
PY


## 3. Revisar y probar la lógica de `soporte_ti.reportes`

La celda contiene el código esencial de `soporte_ti/reportes.py` y una prueba con datos ya procesados.

In [ ]:
%%bash
cd ../src
python - <<'PY'
"""
Módulo reportes del paquete soporte_ti.

Genera salidas legibles a partir de datos procesados por otros módulos.
"""


def generar_linea(resultado: dict) -> str:
    """Genera una línea de reporte para un resultado de comprobación."""
    estado_ip = "IP_OK" if resultado["ip_valida"] else "IP_ERROR"
    estado_puerto = "PUERTO_OK" if resultado["puerto_valido"] else "PUERTO_ERROR"

    return (
        f"{resultado['nombre']:<12} "
        f"{resultado['ip']:<15} "
        f"{resultado['puerto']:<6} "
        f"{estado_ip:<8} "
        f"{estado_puerto}"
    )


def generar_resumen(resultados: list[dict]) -> str:
    """Genera un resumen completo a partir de resultados validados."""
    lineas = [
        "SERVICIO     IP              PUERTO ESTADO_IP ESTADO_PUERTO",
        "-" * 62,
    ]

    for resultado in resultados:
        lineas.append(generar_linea(resultado))

    return "\n".join(lineas)


resultados = [
    {"nombre": "ssh", "ip": "10.0.0.10", "puerto": 22, "ip_valida": True, "puerto_valido": True},
    {"nombre": "api", "ip": "ip_invalida", "puerto": 8080, "ip_valida": False, "puerto_valido": True},
    {"nombre": "backup", "ip": "10.0.0.40", "puerto": 70000, "ip_valida": True, "puerto_valido": False},
]

print(generar_resumen(resultados))
PY


## 4. Ejecutar el ejemplo integrado del paquete

La celda contiene el código completo de `07_paquete_soporte_ti_demo.py` dentro del heredoc. Este código sí importa el paquete real desde `src/soporte_ti`.

In [ ]:
%%bash
cd ../src
python - <<'PY'
"""
Tema 9 - Módulos
Laboratorio 7: ejemplo integrado de paquete propio.

Estructura:
    Tema09/
    └── src/
        ├── 07_paquete_soporte_ti_demo.py
        └── soporte_ti/
            ├── __init__.py
            ├── red.py
            └── reportes.py

Objetivo:
    Usar un paquete propio con varios módulos internos.
"""

from soporte_ti import __version__
from soporte_ti import red
from soporte_ti import reportes


def main():
    """Ejecuta el ejemplo integrado del paquete soporte_ti."""
    print("=== 1. Información del paquete ===")
    print("Paquete soporte_ti versión:", __version__)

    print("\n=== 2. Datos de servicios ===")
    servicios = [
        {"nombre": " SSH ", "ip": "10.0.0.10", "puerto": 22},
        {"nombre": "NGINX", "ip": "10.0.0.20", "puerto": 443},
        {"nombre": "api", "ip": "ip_invalida", "puerto": 8080},
        {"nombre": "backup", "ip": "10.0.0.40", "puerto": 70000},
    ]

    for servicio in servicios:
        print(servicio)

    print("\n=== 3. Comprobar servicios con soporte_ti.red ===")
    resultados = red.comprobar(servicios)

    for resultado in resultados:
        print(resultado)

    print("\n=== 4. Generar resumen con soporte_ti.reportes ===")
    resumen = reportes.generar_resumen(resultados)
    print(resumen)


if __name__ == "__main__":
    main()
PY


## 5. Interpretación

El paquete separa responsabilidades: `red.py` valida y normaliza datos; `reportes.py` genera la salida legible; `__init__.py` define información del paquete, como `__version__`.